# Lesson 0 - Part 3: Vector Stores with ChromaDB

So far you've learned:
- ✅ How the Claude API works
- ✅ What embeddings are (vectors that capture meaning)
- ✅ Why RAG is powerful (give context → Claude reasons)

**Now the practical question: where do we store all these embeddings and documents?**

That's where **vector databases** come in.

In this notebook, we'll:
1. Build our first vector store
2. Add documents to it
3. Query it
4. Understand what's happening under the hood

## Why We Need Vector Databases

Imagine you have 1,000 documents. For each query, you need to:
1. Convert the query to an embedding (a vector)
2. Find the most similar documents

❌ **Naive approach**: Compare query vector to all 1,000 documents
- Time: Very slow (O(n))
- Can't scale past 100K documents

✅ **Vector database approach**: Use specialized indexing
- Time: Fast even for millions
- Smart data structures (HNSW, IVF, LSH)
- Returns similar documents in milliseconds

**ChromaDB** is a lightweight, embedded vector database perfect for learning:
- Runs on your laptop (no cloud)
- Persists to disk
- Good enough for millions of vectors
- Open source

Later, if you need to scale, you can switch to Pinecone or Weaviate. But ChromaDB is where you start.

## Setting Up ChromaDB

Let's start simple: create a vector store and add some data.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

# Create a ChromaDB client (in-memory for now)
client = chromadb.Client()

# Create a collection (like a table in a database)
collection = client.create_collection(name="documents")

print("✅ ChromaDB collection created!")
print(f"Collection name: {collection.name}")
print(f"Collection size: {collection.count()} documents")

/Users/santiagomora/Documents/personal_projects/learn-agentic-rag-end-to-end/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ ChromaDB collection created!
Collection name: documents
Collection size: 0 documents


## Adding Documents

Now let's add some sample documents. Each document needs:
1. **Text**: The actual content
2. **Embedding**: The vector (generated from the text)
3. **Metadata**: Extra info (like source, date)
4. **ID**: Unique identifier

In [3]:
# Load embedding model
# This downloads a small (22MB), fast embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample documents (imagine these came from different sources)
documents = [
    "Machine learning is a subset of artificial intelligence that focuses on learning from data.",
    "Deep learning uses neural networks with multiple layers to learn representations.",
    "Python is a popular programming language for data science and machine learning.",
    "Natural language processing helps computers understand human language.",
    "Computer vision enables machines to interpret visual information from images and videos.",
    "Reinforcement learning trains agents to make decisions through rewards and penalties.",
    "Transfer learning allows models to leverage knowledge from one task to another.",
    "Neural networks are inspired by biological neurons and organized in layers.",
]

# Generate embeddings for all documents
print("🔄 Generating embeddings...")
embeddings = model.encode(documents, show_progress_bar=True)

print(f"\n✅ Generated {len(embeddings)} embeddings")
print(f"Each embedding is a vector of size: {embeddings[0].shape}")
print(f"First 10 values of first embedding: {embeddings[0][:10]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10410.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Generating embeddings...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]


✅ Generated 8 embeddings
Each embedding is a vector of size: (384,)
First 10 values of first embedding: [-0.01861155  0.01348917  0.05557157  0.03718612  0.05443593 -0.02815246
 -0.01594067 -0.04196788 -0.07846537 -0.02188423]


In [4]:
# Add documents to the collection
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),  # Convert numpy array to list
    metadatas=[
        {"source": "ai_primer", "topic": "ML"} for _ in documents
    ],
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print(f"✅ Added {collection.count()} documents to ChromaDB")

✅ Added 8 documents to ChromaDB


## Querying the Vector Store

Now the magic: let's ask a question and see which documents are most similar.

Remember: the query gets converted to an embedding too, then compared to all document embeddings.

In [5]:
# Ask a question
query = "How do neural networks work?"

# Convert query to embedding (same model as documents)
query_embedding = model.encode(query)

# Search for similar documents
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3  # Return top 3 results
)

print(f"\n🔍 Query: '{query}'")
print(f"\n📚 Top 3 most similar documents:\n")

for i, (doc, distance) in enumerate(zip(results['documents'][0], results['distances'][0])):
    print(f"{i+1}. (similarity: {1 - distance:.3f})")
    print(f"   {doc}")
    print()


🔍 Query: 'How do neural networks work?'

📚 Top 3 most similar documents:

1. (similarity: 0.363)
   Neural networks are inspired by biological neurons and organized in layers.

2. (similarity: 0.211)
   Deep learning uses neural networks with multiple layers to learn representations.

3. (similarity: -0.183)
   Computer vision enables machines to interpret visual information from images and videos.



## Understanding Similarity Scores

ChromaDB returns `distances`. The key insight:

- **Lower distance** = more similar
- **Distance 0** = identical
- **Distance 1** = completely different

We can convert to similarity: `similarity = 1 - distance`

So a distance of 0.3 means similarity of 0.7 (70% similar).

In [6]:
# Try different queries and see what gets retrieved
queries = [
    "What is artificial intelligence?",
    "Tell me about Python programming",
    "How does computer vision work?"
]

for query in queries:
    query_emb = model.encode(query)
    results = collection.query(
        query_embeddings=[query_emb.tolist()],
        n_results=2  # Just top 2 this time
    )
    
    print(f"\n🔍 Query: '{query}'")
    for doc, distance in zip(results['documents'][0], results['distances'][0]):
        similarity = 1 - distance
        print(f"   ✓ {similarity:.1%} match: {doc[:70]}...")


🔍 Query: 'What is artificial intelligence?'
   ✓ 22.3% match: Machine learning is a subset of artificial intelligence that focuses o...
   ✓ -17.9% match: Natural language processing helps computers understand human language....

🔍 Query: 'Tell me about Python programming'
   ✓ 58.7% match: Python is a popular programming language for data science and machine ...
   ✓ -36.5% match: Natural language processing helps computers understand human language....

🔍 Query: 'How does computer vision work?'
   ✓ 45.0% match: Computer vision enables machines to interpret visual information from ...
   ✓ -26.4% match: Natural language processing helps computers understand human language....


## Metadata: Adding Context

When we retrieve a document, we don't just want the text—we want to know *where it came from*.

That's what metadata is for.

Let's add richer metadata and see how it helps:

In [7]:
# Create a new collection with better metadata
collection_v2 = client.create_collection(name="docs_with_metadata")

docs_with_meta = [
    {
        "text": "Machine learning is a subset of artificial intelligence.",
        "source": "ai_handbook.pdf",
        "page": 5,
        "section": "Introduction"
    },
    {
        "text": "Deep learning uses neural networks with multiple layers.",
        "source": "ai_handbook.pdf",
        "page": 42,
        "section": "Advanced Topics"
    },
    {
        "text": "Python is popular for data science.",
        "source": "programming_guide.txt",
        "page": None,
        "section": "Languages"
    }
]

# Extract texts and embeddings
texts = [d["text"] for d in docs_with_meta]
embeddings_v2 = model.encode(texts).tolist()
metadatas = [{"source": d["source"], "section": d["section"]} for d in docs_with_meta]

collection_v2.add(
    documents=texts,
    embeddings=embeddings_v2,
    metadatas=metadatas,
    ids=["doc_0", "doc_1", "doc_2"]
)

# Query and see metadata
query = "neural networks"
query_emb = model.encode(query).tolist()

results = collection_v2.query(
    query_embeddings=[query_emb],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

print(f"Query: '{query}'\n")
for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    print(f"✓ Document: {doc}")
    print(f"  Source: {meta['source']}, Section: {meta['section']}")
    print(f"  Similarity: {1-dist:.1%}\n")

Query: 'neural networks'

✓ Document: Deep learning uses neural networks with multiple layers.
  Source: ai_handbook.pdf, Section: Advanced Topics
  Similarity: 12.4%

✓ Document: Machine learning is a subset of artificial intelligence.
  Source: ai_handbook.pdf, Section: Introduction
  Similarity: -20.9%



## Persistence: Saving to Disk

So far we've used in-memory ChromaDB. But if you close the notebook, everything disappears.

Let's use persistent storage instead:

In [8]:
# Create a persistent client (saves to disk)
persistent_client = chromadb.PersistentClient(path="./chroma_data")

# Create collection (will persist across notebook sessions)
persistent_collection = persistent_client.get_or_create_collection(
    name="persistent_docs"
)

# Add documents
test_docs = ["This is persistent", "Data survives restart"]
test_embeddings = model.encode(test_docs).tolist()

persistent_collection.add(
    documents=test_docs,
    embeddings=test_embeddings,
    ids=["persist_1", "persist_2"]
)

print(f"✅ Data saved to ./chroma_data")
print(f"Collection size: {persistent_collection.count()} documents")
print(f"\nClose this notebook and rerun this cell—the data will still be there!")

✅ Data saved to ./chroma_data
Collection size: 2 documents

Close this notebook and rerun this cell—the data will still be there!


In [10]:
# Delete a persistent client from disk
import shutil
shutil.rmtree("./chroma_data")
print("✅ Persistent client deleted from disk")

✅ Persistent client deleted from disk


## When to Upgrade from ChromaDB

ChromaDB is perfect for learning and small-to-medium projects.

But when should you switch?

| Metric | ChromaDB | → Upgrade |
|--------|----------|----------|
| **Documents** | Up to 10M | > 10M |
| **QPS** | ~100 queries/sec | > 100 |
| **Team** | Solo/small team | Enterprise |
| **Uptime SLA** | Development | 99.9%+ |
| **Distribution** | Single machine | Multi-region |

**When to upgrade:**
- Pinecone: Cloud-hosted, scales easily
- Weaviate: Self-hosted or cloud, very flexible
- Milvus: Open source, high-performance
- Qdrant: Fast, modern, good for production

For this tutorial: **ChromaDB is perfect**. We're learning, not scaling.

## Key Takeaway

You now understand:

1. ✅ **What vector databases do**: Store embeddings and retrieve similar ones fast
2. ✅ **How ChromaDB works**: Add docs → Generate embeddings → Query → Get results
3. ✅ **The importance of metadata**: Tells you WHERE the answer came from
4. ✅ **Persistence**: Data survives across sessions

**What's next?**

In Lesson 1, we'll build a complete RAG system:
- Load real documents (from files)
- Chunk them intelligently
- Store in ChromaDB
- Retrieve and pass to Claude
- Get accurate, cited answers

Let's build! 🚀